# EbbRAG — Core Implementation (SPEC-01)

RAG with Spaced Repetition scoring, inspired by Ebbinghaus forgetting curve.

**Stack:** Python vanilla classes · Anthropic API · HuggingFace datasets  
**Runtime:** Google Colab (T4 GPU)

Run cells in order: 1 → 8

## 1. Install dependencies

In [ ]:
!pip install -q anthropic>=0.25.0 sentence-transformers>=2.7.0 rank_bm25>=0.2.2 datasets>=2.19.0 tqdm

## 2. Configure Anthropic API key

Add your key in **Colab Secrets** (🔑 left sidebar):  
`ANTHROPIC_API_KEY = sk-ant-...`

In [ ]:
from google.colab import userdata
import os
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
print('API key loaded ✓')

## 3. Core data classes

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, List
import time, math

@dataclass
class Chunk:
    """Text chunk with Spaced Repetition metadata."""
    id: str
    text: str
    embedding: Optional[List[float]] = None
    stability: float = 0.5          # Ebbinghaus stability S
    last_retrieved: float = field(default_factory=time.time)
    retrieval_count: int = 0

@dataclass
class QueryResult:
    query_id: str
    query: str
    retrieved_chunks: List[Chunk]
    retrieval_scores: List[float]
    parametric_answer: Optional[str]
    final_answer: str
    latency_ms: float
    tokens_used: int
    em: Optional[int] = None

print('Data classes ✓')

## 4. Embedder

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

class Embedder:
    def __init__(self, model_name: str = 'BAAI/bge-small-en-v1.5'):
        print(f'Loading {model_name}...')
        self.model = SentenceTransformer(model_name)
        print('Embedder ✓')

    def embed(self, text: str) -> np.ndarray:
        return self.model.encode(text, normalize_embeddings=True)

    def embed_batch(self, texts: List[str], batch_size: int = 64) -> np.ndarray:
        return self.model.encode(texts, batch_size=batch_size,
                                 normalize_embeddings=True, show_progress_bar=True)

embedder = Embedder()

## 5. ChunkIndex with SR metadata

In [ ]:
import pickle

class ChunkIndex:
    """In-memory vector index with Ebbinghaus SR weighting."""

    def __init__(self, lambda_decay: float = 0.05, epsilon: float = 0.01):
        self.chunks: List[Chunk] = []
        self.embeddings: Optional[np.ndarray] = None
        self.lambda_decay = lambda_decay
        self.epsilon = epsilon

    def add_chunks(self, chunks: List[Chunk], embedder: Embedder):
        texts = [c.text for c in chunks]
        embs = embedder.embed_batch(texts)
        for chunk, emb in zip(chunks, embs):
            chunk.embedding = emb.tolist()
        self.chunks.extend(chunks)
        self.embeddings = np.array([c.embedding for c in self.chunks])
        print(f'Indexed {len(self.chunks)} chunks ✓')

    def sr_weight(self, chunk: Chunk, t_now: float) -> float:
        """Ebbinghaus retention: R = exp(-lambda * delta_t / S)"""
        delta_t_days = (t_now - chunk.last_retrieved) / 86400.0
        retention = math.exp(-self.lambda_decay * delta_t_days / max(chunk.stability, 1e-6))
        return max(retention, self.epsilon)

    def search(self, query_emb: np.ndarray, k: int = 5,
               t_now: Optional[float] = None,
               alpha1: float = 1.0, alpha2: float = 0.0,
               candidate_emb: Optional[np.ndarray] = None,
               alpha3: float = 0.0) -> List[tuple]:
        cosine = self.embeddings @ query_emb
        sr = np.array([self.sr_weight(c, t_now) for c in self.chunks]) if t_now and alpha2 > 0 else np.zeros(len(self.chunks))
        ar = (self.embeddings @ candidate_emb) if candidate_emb is not None and alpha3 > 0 else np.zeros(len(self.chunks))
        combined = alpha1 * cosine + alpha2 * sr + alpha3 * ar
        top_idx = np.argsort(combined)[::-1][:k]
        return [(self.chunks[i], float(combined[i])) for i in top_idx]

    def update_sr(self, chunk_id: str, correct: bool, t_now: float,
                  alpha: float = 0.2, beta: float = 0.8):
        for chunk in self.chunks:
            if chunk.id == chunk_id:
                chunk.stability = min(chunk.stability + alpha, 1.0) if correct else max(chunk.stability * beta, 0.0)
                chunk.last_retrieved = t_now
                chunk.retrieval_count += 1
                break

    def save(self, path: str):
        with open(path, 'wb') as f: pickle.dump(self, f)
        print(f'Saved to {path} ✓')

    @classmethod
    def load(cls, path: str) -> 'ChunkIndex':
        with open(path, 'rb') as f: return pickle.load(f)

print('ChunkIndex ✓')

## 6. Generator (Anthropic)

In [ ]:
import anthropic

class Generator:
    def __init__(self, model: str = 'claude-sonnet-4-6', max_tokens: int = 256):
        self.client = anthropic.Anthropic()
        self.model = model
        self.max_tokens = max_tokens
        self._total_tokens = 0

    def generate(self, query: str, chunks: Optional[List[Chunk]] = None) -> tuple:
        if chunks:
            context = '\n\n'.join([f'[{i+1}] {c.text}' for i, c in enumerate(chunks)])
            prompt = f'Answer the question based on the context.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:'
        else:
            prompt = f'Answer from memory:\n\nQuestion: {query}\nAnswer:'
        resp = self.client.messages.create(
            model=self.model, max_tokens=self.max_tokens,
            messages=[{'role': 'user', 'content': prompt}]
        )
        tokens = resp.usage.input_tokens + resp.usage.output_tokens
        self._total_tokens += tokens
        return resp.content[0].text.strip(), tokens

generator = Generator()
print('Generator ✓')

## 7. EbbRAG main class

In [ ]:
class EbbRAG:
    """
    EbbRAG: RAG with Ebbinghaus Spaced Repetition scoring.
    use_sr=True  → SR-weighted retrieval
    use_ar=True  → Active Recall step (parametric candidate before retrieval)
    """
    def __init__(self, index: ChunkIndex, embedder: Embedder, generator: Generator,
                 k: int = 5, use_sr: bool = True, use_ar: bool = True,
                 alpha1: float = 0.6, alpha2: float = 0.2, alpha3: float = 0.2):
        self.index = index
        self.embedder = embedder
        self.generator = generator
        self.k = k
        self.use_sr = use_sr
        self.use_ar = use_ar
        self.alpha1 = alpha1
        self.alpha2 = alpha2 if use_sr else 0.0
        self.alpha3 = alpha3 if use_ar else 0.0

    def query(self, query: str, query_id: str = 'q0',
              ground_truth: Optional[str] = None,
              t_now: Optional[float] = None) -> QueryResult:
        if t_now is None: t_now = time.time()
        start = time.time()
        total_tokens = 0

        query_emb = self.embedder.embed(query)

        parametric_answer, candidate_emb = None, None
        if self.use_ar:
            parametric_answer, tok = self.generator.generate(query, chunks=None)
            total_tokens += tok
            candidate_emb = self.embedder.embed(parametric_answer)

        results = self.index.search(
            query_emb=query_emb, k=self.k,
            t_now=t_now if self.use_sr else None,
            alpha1=self.alpha1, alpha2=self.alpha2,
            candidate_emb=candidate_emb, alpha3=self.alpha3
        )
        chunks = [c for c, _ in results]
        scores = [s for _, s in results]

        final_answer, tok = self.generator.generate(query, chunks=chunks)
        total_tokens += tok

        if ground_truth is not None:
            correct = ground_truth.lower().strip() in final_answer.lower()
            for chunk in chunks:
                self.index.update_sr(chunk.id, correct=correct, t_now=t_now)

        em = int(ground_truth.lower().strip() in final_answer.lower()) if ground_truth else None

        return QueryResult(
            query_id=query_id, query=query,
            retrieved_chunks=chunks, retrieval_scores=scores,
            parametric_answer=parametric_answer, final_answer=final_answer,
            latency_ms=(time.time() - start) * 1000,
            tokens_used=total_tokens, em=em
        )

print('EbbRAG ✓')

## 8. Smoke test

In [ ]:
toy_chunks = [
    Chunk(id='c1', text='Hermann Ebbinghaus was a German psychologist who studied memory and forgetting.'),
    Chunk(id='c2', text='The forgetting curve shows how memory retention decreases over time without reinforcement.'),
    Chunk(id='c3', text='Spaced repetition exploits the spacing effect to improve long-term memory retention.'),
]

index = ChunkIndex(lambda_decay=0.05)
index.add_chunks(toy_chunks, embedder)

ebb = EbbRAG(index=index, embedder=embedder, generator=generator, k=2, use_sr=True, use_ar=True)
result = ebb.query('Who studied the forgetting curve?', query_id='smoke-01', ground_truth='Ebbinghaus')

print(f'Query:             {result.query}')
print(f'Parametric answer: {result.parametric_answer}')
print(f'Final answer:      {result.final_answer}')
print(f'Exact Match:       {result.em}')
print(f'Latency:           {result.latency_ms:.0f}ms')
print(f'Tokens used:       {result.tokens_used}')
print()
for chunk, score in zip(result.retrieved_chunks, result.retrieval_scores):
    print(f'  [{score:.3f}] {chunk.text[:70]}')
print()
print('Smoke test complete ✓')